# ADJSCC vs ReJSCC on speech — Colab training

Runs the phase-3 experiment matrix from `docs/AUDIO_PLAN.md`: does
**feature-pooled** SNR conditioning (ADJSCC's AF module) beat **SNR-only**
conditioning (ReJSCC's regulating module) when the source is speech?

ReJSCC showed pooled features are unnecessary — but only on images. Their
speech experiments have no ADJSCC arm at all, which is the gap this fills.

Six arms, identical in every respect except the two axes under test:

| run | `cond` | training SNR |
|---|---|---|
| ADJSCC arm | `af` | U[0,20] dB |
| ReJSCC arm | `reg` | U[0,20] dB |
| BDJSCC | `none` | fixed 4 dB |
| BDJSCC | `none` | fixed 7 dB |
| BDJSCC | `none` | fixed 13 dB |
| BDJSCC | `none` | U[0,20] dB |

Baselines match ReJSCC section 4.3 so results drop into their table.

## 1. Setup

Clones the repo and installs dependencies when running on Colab; a no-op
if you already opened this notebook inside a local checkout.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('ADJSCC_REPO', 'https://github.com/prashantrajbista/wireless_image_transmission.git')

if IN_COLAB:
    if not pathlib.Path('wireless_image_transmission').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
    os.chdir('wireless_image_transmission')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'soundfile', 'pystoi', 'pesq', 'pytorch-msssim',
                    'huggingface_hub', 'pyarrow', 'wandb'], check=True)
else:
    # running from notebooks/ inside a checkout
    if pathlib.Path.cwd().name == 'notebooks':
        os.chdir('..')

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
import torch
from adjscc.engine import default_args, train, pick_device

DEVICE = pick_device()
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu':
    print('WARNING: no GPU. On Colab: Runtime > Change runtime type > GPU.')

# DataLoader workers: Colab (Linux, fork) is happy with 2; notebooks on
# macOS/Windows use spawn and need 0 or they deadlock on import.
WORKERS = 2 if sys.platform.startswith('linux') else 0

### Weights & Biases (optional)

Skip this cell to train without logging — `train()` degrades to printing.
Every run records its full dataset provenance (corpus, source, sample rate,
clip length, how many clips were dropped) alongside the usual hyperparameters,
so two runs can be checked for comparability from the config alone.

In [ ]:
USE_WANDB = False  # set True and run this cell to enable logging

if USE_WANDB:
    import wandb
    wandb.login()  # paste your key when prompted
    os.environ.setdefault('WANDB_PROJECT', 'wireless-image-transmission')

## 2. Data

VoiceBank-DEMAND from the HuggingFace mirror
([JacobLinCool/VoiceBank-DEMAND-16k](https://huggingface.co/datasets/JacobLinCool/VoiceBank-DEMAND-16k)):
11,572 train / 824 test utterances, the split DeepSC-S describes as "more
than 10,000 trainset and 800 testset". We extract the **clean** column —
this is a reconstruction task, so the source signal is clean speech.

~2.3 GB download, then a byte-copy extraction to wav. Idempotent.

In [ ]:
from adjscc.audio_data import fetch_voicebank

DATA_ROOT = 'data/voicebank'
fetch_voicebank(DATA_ROOT)

### Sample rate — a real choice, not a detail

The 128x128 framing fixes the clip at 16,384 samples, so the sample rate
*is* the clip duration:

| `SR` | clip | VoiceBank kept | matches |
|---|---|---|---|
| 8000 | 2.048 s | **66%** (short utterances dropped) | ReJSCC |
| 16000 | 1.024 s | **100%** | DeepSC-S |

Bandwidth ratio and framing are identical either way. 8 kHz is the default
because it matches the arm being contested, but it discards a third of the
corpus and the discard is length-biased. Whichever you pick, keep it the
same across all six runs.

In [ ]:
SR = 8000        # or 16000 to keep the whole corpus
RATIO = 0.5      # R = C/32, so C=16 — ReJSCC's speech operating point
FILTERS = 256    # drop to 64 for a fast sanity pass
EPOCHS = 2000    # ReJSCC's speech setting; cut hard for a first run
BATCH = 256

## 3. Sanity check before committing GPU hours

One short run, then **listen to the output**. A model that trains to a
plausible-looking SDR can still be emitting rectified garbage; ears catch
in seconds what a metric hides.

In [ ]:
sanity = default_args(
    modality='audio', cond='af', ratio=RATIO, sr=SR,
    filters=64, epochs=3, batch=BATCH, workers=WORKERS,
    optimizer='rmsprop', lr=1e-3, lr_decay=0.999,
    data_root=DATA_ROOT, out='ckpt/audio_sanity.pt', no_wandb=True)
train(sanity)

In [ ]:
import numpy as np, torch
from IPython.display import Audio, display
from adjscc.engine import load_model
from adjscc.audio_data import loaders_audio, to_waveform

model, ck = load_model('ckpt/audio_sanity.pt', DEVICE)
_, test_loader = loaders_audio(8, root=DATA_ROOT, sr=SR, workers=0)
x, _ = next(iter(test_loader))
x = x[:1].to(DEVICE)
with torch.no_grad():
    y = model(x, 10.0)

print('reference');     display(Audio(to_waveform(x.cpu())[0].numpy(), rate=SR))
print('reconstruction @ 10 dB'); display(Audio(to_waveform(y.cpu())[0].numpy(), rate=SR))
print('output range:', float(y.min()), float(y.max()), '(must span negative)')

## 4. The six arms

Sequential, because they share one GPU. Each writes its own checkpoint, so
an interrupted session resumes by re-running with the finished arms removed
from `RUNS`.

**Every arm must use identical epochs, optimizer and sample rate.** The
comparison is void otherwise — that is the whole point of the experiment.

In [ ]:
RUNS = [
    dict(name='audio_af',     cond='af',   snr_fixed=None),
    dict(name='audio_reg',    cond='reg',  snr_fixed=None),
    dict(name='audio_bd4',    cond='none', snr_fixed=4.0),
    dict(name='audio_bd7',    cond='none', snr_fixed=7.0),
    dict(name='audio_bd13',   cond='none', snr_fixed=13.0),
    dict(name='audio_bduni',  cond='none', snr_fixed=None),
]

results = {}
for r in RUNS:
    out = f"ckpt/{r['name']}.pt"
    if pathlib.Path(out).exists():
        print(f"skip {r['name']} (checkpoint exists)")
        continue
    print(f"\n=== {r['name']} " + '=' * 40)
    args = default_args(
        modality='audio', cond=r['cond'], snr_fixed=r['snr_fixed'],
        ratio=RATIO, sr=SR, filters=FILTERS, epochs=EPOCHS, batch=BATCH,
        workers=WORKERS, optimizer='rmsprop', lr=1e-3, lr_decay=0.999,
        data_root=DATA_ROOT, out=out, no_wandb=not USE_WANDB)
    results[r['name']] = train(args)

results

## 5. Evaluate across the SNR range

`--perceptual` adds STOI and PESQ. They are per-utterance and CPU-bound, so
they are off by default; with `REPEATS=1` they are affordable.

In [ ]:
REPEATS = 1        # paper uses 10; raise once the trend is visible
PERCEPTUAL = True
SNRS = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20]

for r in RUNS:
    ckpt = f"ckpt/{r['name']}.pt"
    if not pathlib.Path(ckpt).exists():
        continue
    subprocess.run([sys.executable, 'scripts/eval.py', '--ckpt', ckpt,
                    '--snr-list', ','.join(map(str, SNRS)),
                    '--repeats', str(REPEATS), '--sr', str(SR),
                    '--data-root', DATA_ROOT,
                    '--out', f"results/{r['name']}.csv"]
                   + (['--perceptual'] if PERCEPTUAL else []), check=True)

## 6. The plot that answers the question

If the `af` curve sits above `reg`, pooled features earn their keep on
speech and ReJSCC's claim does not generalize across modalities. If they
overlap, it does. **Both outcomes are a result** — the experiment is
informative either way, which is why it is worth running.

In [ ]:
import csv
import matplotlib.pyplot as plt

def load(name):
    p = pathlib.Path(f'results/{name}.csv')
    if not p.exists():
        return None
    with open(p) as f:
        rows = list(csv.DictReader(f))
    return ([float(r['snr']) for r in rows],
            [float(r['sdr']) for r in rows])

STYLE = {'audio_af': ('ADJSCC (pooled feats + SNR)', 'o-', 2.2),
         'audio_reg': ('ReJSCC (SNR only)', 's-', 2.2),
         'audio_bduni': ('BDJSCC uniform', '^--', 1.2),
         'audio_bd4': ('BDJSCC @4 dB', ':', 1.0),
         'audio_bd7': ('BDJSCC @7 dB', ':', 1.0),
         'audio_bd13': ('BDJSCC @13 dB', ':', 1.0)}

plt.figure(figsize=(7, 5))
for name, (label, fmt, lw) in STYLE.items():
    d = load(name)
    if d:
        plt.plot(d[0], d[1], fmt, label=label, linewidth=lw)
plt.xlabel('test SNR (dB)'); plt.ylabel('SDR (dB)')
plt.title(f'Speech JSCC, R={RATIO}, {SR//1000} kHz, AWGN')
plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig('graphs/audio_sdr_vs_snr.png', dpi=150)
plt.show()

## 7. Listen to the trained arms

The final check. Metrics rank; ears tell you whether either is usable.

In [ ]:
for name in ('audio_af', 'audio_reg'):
    ckpt = f'ckpt/{name}.pt'
    if not pathlib.Path(ckpt).exists():
        continue
    m, _ = load_model(ckpt, DEVICE)
    with torch.no_grad():
        for snr in (0.0, 10.0, 20.0):
            rec = m(x, snr)
            print(f'{name} @ {snr:g} dB')
            display(Audio(to_waveform(rec.cpu())[0].numpy(), rate=SR))